# Scenario 1: <sup>1</sup>H-NMR kinetic analysis of Pd(0)-catalyzed cycloisomerization

## Data Processing and Analysis Notebook

---

Raw Bruker FIDs of the cycloisomerization reaction are processed with NMRPy, the educt resonances are integrated numerically against the mesitylene internal standard, and the resulting concentration time course is written into an EnzymeML document.

The reaction was measured at three ligand loadings (2, 5 and 33 equivalents of BINOL relative to Pd(0)), which are three measurements in one EnzymeML document: `m1` = 2 eq, `m2` = 5 eq and `m3` = 33 eq.

**This notebook is run once per ligand loading.** Set `DATASET` below to `"2eq"`, then `"5eq"`, then `"33eq"`, and run the whole notebook each time. Each run fills one measurement and prepares the next one, so the document accumulates m1 → m2 → m3 across the three runs.

This notebook writes its results to `data/run/`, so the published results in `data/output/` are never overwritten. The modeling notebook uses the published results by default.

Everything that differs between the three datasets is collected in the `DATASETS` table further below, the rest of the notebook is identical for all three.

### Notebook Setup

The essential packages to work with NMRPy and the EnzymeML extension are:

- `matplotlib`
- `nmrpy`
- `pyenzyme`

This notebook additionally uses:

- `numpy` and `scipy`, for the numeric integration in `modules/`
- `rich`, for readable console output

The helper functions in `modules/` keep the notebook readable; each module documents what it does.

Interactive plotting requires either the `TkAgg` or `ipympl` backend. While the `ipympl` backend allows for convenient interactive inline plotting in the Jupyter notebook, it does not run as smoothly as the `TkAgg` backend that pops up the graphs in separate windows. If interactivity is not required, `matplotlib.use("ipympl")` can be commented out and uncommenting `%matplotlib inline` will plot static graphs inline instead.

In [ ]:
import os
import shutil
from rich import print

import matplotlib
from matplotlib import pyplot as plt

import nmrpy
import pyenzyme as pe

plt.ioff()
matplotlib.use("ipympl")
# %matplotlib inline

In [ ]:
from modules.assignment import assign_species
from modules.io import (
    save_data_model,
    set_concentrations,
    summarize_measurement,
    write_concentrations_csv,
)
from modules.numeric_integration import (
    calculate_educt_concentration_from_integrals,
    integrate_fid_array,
    integrate_simpson,
    local_linear_baseline,
)
from modules.phasing import apply_phase_angles
from modules.quantification import (
    QuadratureRules,
    local_linear_baseline_record,
    record_numeric_integration,
)

---

### Selecting the dataset

**This is the only cell that changes between runs.** Set `DATASET` to `"2eq"`, `"5eq"` or `"33eq"`, in that order, and run the notebook from top to bottom.

In [ ]:
DATASET = "2eq"  # "2eq" -> "5eq" -> "33eq", in that order

#### Per-dataset parameters

Every value below was determined interactively once and is recorded here, so that the notebook runs without any manual step. The fields are:

- `array`: the raw Bruker directory under `data/input/`. The companion `*_t0/` directories hold t = 0 spectra acquired separately with a wider spectral width (to also capture the hydride region); they are not part of these arrays.
- `drop_fids`: spectra without a recorded reaction time, excluded before processing.
- `p0` / `p1`: zero- and first-order phase angles, one pair per FID, in array order.
- `picks`: peaks and whole-multiplet ranges per FID, in ppm order: mesitylene singlet, educt doublet 1, educt doublet 2.
- `times`: reaction time of each spectrum in hours. The acquisition timestamps in the Bruker files do not reflect the reaction times, so these were recorded separately.
- `t0`: educt concentration at t = 0 where it was determined separately (from the `*_t0` spectra) rather than from this array; `None` where the first spectrum of the array is already at t = 0.
- `next_measurement`: metadata for the measurement this run prepares for the next; `None` on the last run.

In [ ]:
DATASETS = {
    "2eq": {
        "equivalents": '2 eq',
        "measurement": 'm1',
        "array": 'fid_array_conc_2',
        "drop_fids": ['fid3'],
        "p0": [360.73274724010116, 356.22598438618013, 360.56113398576633, 357.7438183114879, 361.989253325503, 359.4166363874218],
        "p1": [-340.4391966295698, -340.96475853483366, -340.97443742922854, -341.2037088062441, -340.9265350800337, -341.0338195633258],
        "picks": {
            "fid0": ([6.71, 3.14, 3.13, 3.03, 3.02], [[6.76, 6.69], [3.17, 3.1], [3.05, 2.99]]),
            "fid1": ([6.71, 3.15, 3.13, 3.03, 3.02], [[6.75, 6.69], [3.17, 3.11], [3.05, 2.99]]),
            "fid2": ([6.71, 3.15, 3.13, 3.03, 3.02], [[6.75, 6.7], [3.17, 3.11], [3.05, 2.99]]),
            "fid4": ([6.71, 3.15, 3.13, 3.03, 3.02], [[6.75, 6.69], [3.17, 3.11], [3.05, 3.01]]),
            "fid5": ([6.71, 3.14, 3.04], [[6.74, 6.69], [3.15, 3.13], [3.05, 3.01]]),
            "fid6": ([6.71, 3.14, 3.04], [[6.75, 6.69], [3.15, 3.13], [3.05, 3.01]]),
        },
        "times": [2.5, 3.5, 5.5, 9.0, 19.0, 40.0],
        "t0": 0.131337649,
        "next_measurement": {
            "id": 'm2',
            "name": 'S-BINOL 2mg Pd(PCy3)2 5mg - C6D6 - 80C',
            "prepared": {
                's1': 0.11986,
                's3': 0.005994,
                's4': 0.03353,
                's5': 0.15309,
                's6': 5.644682115270349e-06,
            },
        },
    },
    "5eq": {
        "equivalents": '5 eq',
        "measurement": 'm2',
        "array": 'fid_array_conc_5',
        "drop_fids": [],
        "p0": [356.32534954246955, 360.0517385507509, 357.9000996845492, 361.41427131083947, 357.6349353436972, 358.4128930342035, 356.31243592939734, 357.4108531612718],
        "p1": [-341.7373640253088, -342.0458241926064, -342.29621179569165, -342.31661689072644, -342.31404103369675, -342.4071759099726, -342.19663000975936, -342.21640964096264],
        "picks": {
            "fid0": ([6.71, 3.14, 3.12, 3.03, 3.02], [[6.75, 6.69], [3.17, 3.1], [3.06, 2.99]]),
            "fid1": ([6.71, 3.14, 3.12, 3.03, 3.02], [[6.74, 6.69], [3.17, 3.1], [3.04, 2.99]]),
            "fid2": ([6.71, 3.14, 3.12, 3.03, 3.02], [[6.75, 6.69], [3.17, 3.1], [3.05, 2.99]]),
            "fid3": ([6.71, 3.14, 3.12, 3.03], [[6.75, 6.69], [3.15, 3.11], [3.05, 3.01]]),
            "fid4": ([6.71, 3.14, 3.12, 3.03], [[6.75, 6.69], [3.15, 3.11], [3.05, 3.01]]),
            "fid5": ([6.71, 3.14, 3.12, 3.03], [[6.75, 6.69], [3.15, 3.11], [3.05, 3.01]]),
            "fid6": ([6.71, 3.14, 3.12, 3.03], [[6.75, 6.68], [3.15, 3.11], [3.05, 3.01]]),
            "fid7": ([6.71, 3.14, 3.12, 3.03], [[6.75, 6.69], [3.15, 3.11], [3.05, 3.01]]),
        },
        "times": [0.8333333333333334, 3.83333333333333, 7.833333333333333, 14.333333333333334, 16.5833333333333, 21.3333333333333, 32.3333333333333, 39.0],
        "t0": 0.108466994,
        "next_measurement": {
            "id": 'm3',
            "name": 'S-BINOL 2mg Pd(PCy3)2 33mg - C6D6 - 80C',
            "prepared": {
                's1': 0.12003,
                's3': 0.006594,
                's4': 0.18441,
                's5': 0.15475,
                's6': 5.644682115270349e-06,
            },
        },
    },
    "33eq": {
        "equivalents": '33 eq',
        "measurement": 'm3',
        "array": 'fid_array_conc_33',
        "drop_fids": [],
        "p0": [315.75, 405.65, 350.7, 349.8, 350.9, 346.1],
        "p1": [-278.562564, -401.250155, -334.9, -324.4, -325.1, -315.6],
        "picks": {
            "fid0": ([6.71, 3.15, 3.13, 3.03], [[6.78, 6.68], [3.18, 3.09], [3.07, 2.99]]),
            "fid1": ([6.71, 3.14, 3.12, 3.02], [[6.74, 6.68], [3.17, 3.1], [3.04, 2.99]]),
            "fid2": ([6.72], [[6.74, 6.69]]),
            "fid3": ([6.71], [[6.74, 6.69]]),
            "fid4": ([6.71], [[6.74, 6.69]]),
            "fid5": ([6.71], [[6.74, 6.68]]),
        },
        "times": [0.0, 2.0, 8.0, 14.33, 14.33, 19.08],
        "t0": None,
        "next_measurement": None,
    },
}

In [ ]:
CFG = DATASETS[DATASET]
CURRENT_MEASUREMENT = CFG["measurement"]

print(f"Processing {DATASET} ({CFG['equivalents']} BINOL to Pd(0)) -> measurement {CURRENT_MEASUREMENT}")

---

### Processing of the NMR Data

#### NMR data loading

An NMRPy workflow starts with loading the NMR data. NMRPy supports a variety of file formats, including Bruker and Varian. FIDs without a recorded reaction time are removed straight away, so that the phase angles, ranges and times all line up with the same set of spectra.

In [ ]:
path_to_data = f"./data/input/{CFG['array']}"

p = nmrpy.from_path(path_to_data)

for fid_id in CFG["drop_fids"]:
    p.del_fid(fid_id)
    print(f"Excluded {fid_id}: no reaction time recorded for it.")

print(f"{len(p.get_fids())} spectra: {[fid.id for fid in p.get_fids()]}")

#### EnzymeML document loading

The EnzymeML document is loaded into NMRPy with `parse_enzymeml_document`, which links its species and measurement metadata with the NMRPy data model.

The template in `data/input/cycloisomerization.json` describes the reaction, its species and the first measurement (`m1`), without any time-course data. The 2 eq run starts a new document in `data/run/` from this template; the 5 eq and 33 eq runs continue the document written by the run before them.

The last measurement in the document is the one to fill. It is checked against the measurement expected for `DATASET`, so running the datasets out of order fails immediately instead of writing a time course into the wrong measurement.

In [ ]:
path_to_template = "./data/input/cycloisomerization.json"
RUN_DIR = "./data/run"
os.makedirs(RUN_DIR, exist_ok=True)
path_to_enzymeml = f"{RUN_DIR}/cycloisomerization.json"

if CURRENT_MEASUREMENT == "m1":
    shutil.copy(path_to_template, path_to_enzymeml)
    print(f"Started a new document from {path_to_template}")

p.parse_enzymeml_document(path_to_enzymeml)

latest = p.enzymeml_document.measurements[-1].id
if latest != CURRENT_MEASUREMENT:
    raise ValueError(
        f"The document is ready for {latest!r}, but DATASET={DATASET!r} expects "
        f"{CURRENT_MEASUREMENT!r}. Run the datasets in order: 2eq, 5eq, 33eq."
    )

print(f"Filling measurement {CURRENT_MEASUREMENT}")

#### Apodization, zero-filling, and Fourier transformation

It is common to improve the signal-to-noise ratio of NMR spectra by applying apodization. Also, the data has to be transformed into the frequency domain by Fourier transformation.

In [ ]:
p.emhz_fids(lb=0.3)     # Apodization
p.zf_fids()             # Zero-filling
p.ft_fids()             # Fourier transformation

In [ ]:
p.plot_array()

#### Phase correction

Every FID is phased with its own recorded pair of phase angles instead of by the automatic `phase_correct_fids()` method. Recording the angles makes the processing reproducible and reportable, independent of an optimizer being re-run.

For 2 eq and 5 eq, the recorded angles are those determined by `phase_correct_fids()`, with 180° added to the zero-order angle: the automatic fit does not tell an upright spectrum from an inverted one, and it returned these spectra inverted. On the 33 eq array the automatic fit does not converge usefully, because the educt signals drop close to the noise floor within the first few time points; these spectra were phased manually.

To determine phase angles interactively instead, run `p.fid0.phaser()` in a cell of its own.

In [ ]:
apply_phase_angles(p, CFG["p0"], CFG["p1"], plot=False)

#### Realization and normalization

The imaginary part of the spectra is discarded, and the spectra are normalized by the maximum data value among all spectra.

In [ ]:
p.real_fids()           # Discard imaginary part
p.norm_fids()           # Normalization

In [ ]:
p.plot_array()

#### Calibration

The spectra likely need to be calibrated by setting the ppm of a reference peak, which can be done using the `calibrate` method. In this scenario, the spectra are already calibrated, so this step is skipped.

#### Baseline correction

NMRPy offers a global baseline correction (`baseliner_fids()` to select baseline regions, then `baseline_correct_fids()` to fit and subtract a polynomial through them), which removes broad roll and offset across the whole spectrum. **It is deliberately not applied here.** On these spectra it changes the resulting concentrations by at most 0.00015 %, because the spectra are already flat on this scale and each integrated range gets its own local baseline right before integration (see *Numeric integration*), which is what matters.

Skipping the global correction also keeps the notebook free of manual steps: `baseliner_fids()` is an interactive widget, and without selected regions `baseline_correct_fids()` leaves the data unchanged. The saved NMRPy data model records correctly that no global baseline correction was applied.

To apply it anyway, run `p.baseliner_fids()` in a cell of its own, select the regions, and then call `p.baseline_correct_fids()`.

#### Peak picking

The relevant resonances are:

- ≈6.71 ppm: Mesitylene singlet (internal standard, 3 protons)
- ≈3.14–3.13 ppm: Educt doublet 1 (2 protons)
- ≈3.03–3.02 ppm: Educt doublet 2 (2 protons)

Each range spans a **whole multiplet** rather than an individual line, because the whole range is integrated below. The educt triplet at ≈5.12–5.08 ppm is not used: the two doublets alone determine the educt concentration, and the triplet overlaps with neighboring product signals.

In the 33 eq array, `fid2` to `fid5` carry the mesitylene range only. This is deliberate: the reaction is complete before the third time point, so there is no educt resonance left to pick, and picking noise in those windows would produce a false non-zero tail.

To pick peaks interactively instead, run `p.peakpicker()` or `p.fid0.peakpicker()` in a cell of its own.

In [ ]:
for fid_id, (peaks, ranges) in CFG["picks"].items():
    fid = getattr(p, fid_id)
    fid.peaks = peaks
    fid.ranges = ranges
    print(f"{fid_id}: peaks={peaks} ranges={ranges}")

#### Peak assignment

The relevant species are:

- s5: Mesitylene
- s1: Educt

Species are usually assigned to peaks with the interactive `assign_peaks()` widget. As the assignment here is purely positional, `assign_species()` from `modules/` reproduces it without manual input: the ranges are picked in ppm order, so the first peak is always the mesitylene singlet and every later peak belongs to the educt. It also creates the `Peak` objects of the NMRPy data model, in which the integration is recorded further below.

To assign species interactively instead, run `p.assign_peaks()` in a cell of its own.

In [ ]:
assign_species(p, standard_species="s5", educt_species="s1", verbose=True)

#### Numeric integration

Each range is integrated numerically with Simpson's rule (`scipy.integrate.simpson`), after subtracting a local linear baseline anchored on the mean of the outer three data points at each edge of the range. The local baseline is applied identically to every range of every FID, including the mesitylene reference.

NMRPy's Lorentzian/Gaussian deconvolution is not used. Fitting peak shapes becomes unreliable as the peaks approach the noise floor, which is exactly the low-concentration regime that determines the late part of the time course, and the fit has no notion of a local baseline offset, so it tends to overestimate peak areas at low concentration.

The choice of quadrature rule barely matters on data this densely sampled: rectangle, trapezoidal and Simpson's rule agree to within 0.2 %. The local baseline, in contrast, lowers the educt integrals by 8–91 % and the resulting concentrations by up to 70 %.

In [ ]:
simpson_integrals = integrate_fid_array(p, integrate_simpson, baseline=local_linear_baseline)

for fid_id, integrals in simpson_integrals.items():
    print(fid_id, [round(i, 5) for i in integrals])

##### Recording the integration in the NMRPy data model

Every `Peak` in the NMRPy data model carries a `peak_quantification` describing **how** its area was obtained: the method (numeric integration), the quadrature rule, and the local baseline that was subtracted, including its anchor points, slope and intercept. This makes the integration traceable from the saved data model alone.

In [ ]:
record_numeric_integration(
    p,
    simpson_integrals,
    QuadratureRules.SIMPSON,
    local_linear_baseline_record,
)

---

### Analysis of the cycloisomerization reaction

#### Concentration calculation

The integrals of the two educt doublets are summed and scaled against the integral of the mesitylene internal standard, weighted by the number of protons of each signal, and multiplied by the prepared mesitylene concentration recorded in the EnzymeML document.

As the time stamps in the Bruker raw data do not reflect the reaction times, the time array is set from `times`.

In [ ]:
p._params["acqtime_array"] = CFG["times"]

if len(p.get_fids()) != len(CFG["times"]):
    raise ValueError(f"{len(p.get_fids())} spectra but {len(CFG['times'])} reaction times.")

concentrations = calculate_educt_concentration_from_integrals(
    p, simpson_integrals, p.enzymeml_document, CURRENT_MEASUREMENT
)

print(f"Time array:          {p.t}")
print(f"Concentration array: {concentrations['s1']}")

##### Baseline floor correction

Even with the local baseline subtracted, the educt ranges do not integrate to exactly zero once the educt is consumed: a small, constant residual level remains. This baseline floor is the mean of two levels. The 2 eq level, 0.0051676 M, is the mean of the last two 2 eq concentrations before this correction. The 33 eq level, 0.005161 M, was determined in an earlier NMRPy analysis from the last four 33 eq spectra, which this notebook does not integrate (see *Peak picking*). The two agree to within 0.13 % despite an 11-fold difference in ligand loading. Their mean, **0.005164 M**, is subtracted from every integrated educt concentration of all three datasets, including the separately determined initial concentration. The 5 eq dataset was not used for this determination, as its residual level is markedly higher.

Subtracting a constant can push values that are already at the floor slightly below zero, for example the last 2 eq point. These values are kept as they are rather than clipped, so that the correction is identical for every point.

Spectra in which only the mesitylene peak is picked contain no educt signal: in the last four 33 eq spectra the educt is used up (see *Peak picking*). Their concentration of 0 is not integrated, so it is left as it is.

In [ ]:
BASELINE_FLOOR = 0.005164  # mol/l

# Spectra with only the mesitylene range picked hold no educt signal; their 0 stays 0.
integrated = [len(fid.ranges) > 1 for fid in p.get_fids()]
concentrations["s1"] = [
    c - BASELINE_FLOOR if has_educt else c
    for c, has_educt in zip(concentrations["s1"], integrated)
]
t0 = None if CFG["t0"] is None else CFG["t0"] - BASELINE_FLOOR

print(f"Concentration array: {concentrations['s1']}")
if t0 is not None:
    print(f"Initial concentration at t = 0: {t0}")

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)
ax.plot(p.t, concentrations["s1"], "s", mec="k", label="s1")
ax.legend()
ax.set_xlabel("Time (h)")
ax.set_ylabel("Concentration (M)")
ax.set_title(f"{CFG['equivalents']} BINOL to Pd(0) ({CURRENT_MEASUREMENT})")
plt.show()

#### Apply the concentrations to EnzymeML

The concentrations are attached to the FID array with `set_concentrations()` from `modules/`. NMRPy's own `FidArray.concentrations` setter checks the species against the **last** FID of the array only, and so rejects a valid educt series whenever the last spectrum carries fewer assigned species than the first, which is the case once the reaction has run to completion. The helper applies the same checks across every FID instead.

In [ ]:
set_concentrations(p, concentrations)

The concentrations are then written into the EnzymeML document with `FidArray.apply_to_enzymeml()`.

Where the array does not start at t = 0 (2 eq and 5 eq), the initial concentration was determined separately from the corresponding `*_t0` spectrum and is added with `add_t0_to_enzymeml(use_t1=False, t0={...})`. Where the first spectrum is already at t = 0 (33 eq), `use_t1=True` only shifts the time axis so that it starts at zero.

The time course, including the point at t = 0, is also written to a CSV file in `data/run/`.

In [ ]:
enzymeml_doc = p.apply_to_enzymeml(measurement_id=CURRENT_MEASUREMENT)

if t0 is None:
    p.add_t0_to_enzymeml(gui=False, measurement_id=CURRENT_MEASUREMENT, use_t1=True)
else:
    p.add_t0_to_enzymeml(
        gui=False,
        measurement_id=CURRENT_MEASUREMENT,
        use_t1=False,
        t0={"s1": t0},
    )

write_concentrations_csv(
    p, CURRENT_MEASUREMENT, directory=RUN_DIR, enzymeml_document=enzymeml_doc
)
summarize_measurement(enzymeml_doc, CURRENT_MEASUREMENT)

#### Prepare the next measurement

`FidArray.create_new_enzymeml_measurement()` appends a new measurement to the document, using the one just filled as a template and overriding what differs: the ID, the name, and the prepared concentrations. pH and temperature are the same for all three measurements and are kept. This lets the notebook be run again for the next ligand loading without editing the EnzymeML document by hand. On the last run, `next_measurement` is `None` and nothing is prepared.

In [ ]:
nxt = CFG["next_measurement"]

if nxt is None:
    print(f"{CURRENT_MEASUREMENT} is the last measurement; nothing to prepare.")
else:
    p.create_new_enzymeml_measurement(
        gui=False,
        template_measurement=True,
        template_id=CURRENT_MEASUREMENT,
        keep_ph=True,
        keep_temperature=True,
        keep_prepared=False,
        id=nxt["id"],
        name=nxt["name"],
        prepared=nxt["prepared"],
    )
    print(f"Prepared {p.enzymeml_document.measurements[-1].id}: {p.enzymeml_document.measurements[-1].name}")

print(f"Document now holds: {[m.id for m in p.enzymeml_document.measurements]}")

In [ ]:
pe.write_enzymeml(p.enzymeml_document, path=path_to_enzymeml)

#### Save NMRPy data model

The NMRPy data model for the current measurement is saved as a JSON file, too. It holds all relevant NMR data and metadata in a structured format, including the processing steps and the `peak_quantification` records of the numeric integration.

In [ ]:
save_data_model(p, CURRENT_MEASUREMENT, directory=RUN_DIR)

#### Save the NMRPy library state

The entire state of the NMRPy library, including all loaded data and calculated results, can be saved in a pickle file, allowing to easily reload the state at a later point in time without having to repeat the data loading and processing steps.

In [ ]:
p.save_to_file(f"{RUN_DIR}/{CURRENT_MEASUREMENT}.nmrpy", overwrite=True)

---

### Next run

Set `DATASET` to the next entry and run the notebook again from the top. After all three runs, `data/run/cycloisomerization.json` holds m1, m2 and m3 with their complete concentration time courses, ready for the modeling notebook.

---

### Disclosure

**Contributions**

If you wish to contribute to the EnzymeML and/or NMRPy platforms, find us on our [EnzymeML GitHub](https://github.com/EnzymeML) and [NMRPy GitHub](https://github.com/NMRPy)!

**BSD 3-Clause License**

Copyright (c) 2026 Torsten Giess

Redistribution and use in source and binary forms, with or without modification, are permitted provided that the following conditions are met:

1. Redistributions of source code must retain the above copyright notice, this list of conditions and the following disclaimer.

2. Redistributions in binary form must reproduce the above copyright notice, this list of conditions and the following disclaimer in the documentation and/or other materials provided with the distribution.

3. Neither the name of the copyright holder nor the names of its contributors may be used to endorse or promote products derived from this software without specific prior written permission.

THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS “AS IS” AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY, OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.